# Task 3: News Sentiment and Stock Movement Correlation

This notebook scores financial-news headline sentiment, aligns articles to trading days, computes daily adjusted-close returns, and measures whether average daily sentiment is statistically associated with same-day stock movement.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yfinance as yf

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.correlation import (
    average_return_by_sentiment,
    interpret_correlation,
    pearson_correlation,
    sentiment_return_dataset,
)
from src.data_loading import load_news, load_price_data, normalize_price_data
from src.sentiment import add_sentiment_scores
from src.visualization import plot_sentiment_return_scatter, save_current_figure

sns.set_theme(style='whitegrid')
RAW = PROJECT_ROOT / 'data' / 'raw'
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

## Methodology

- Use TextBlob polarity because it is lightweight, deterministic, and works well for short headline text without external model downloads.
- Normalize news timestamps to calendar days, then align each article to the same or next available stock trading day.
- Compute adjusted-close daily return as `(Adj Close_t - Adj Close_t-1) / Adj Close_t-1 * 100`.
- Average multiple same-stock headlines published on the same trading day before calculating Pearson correlation.

## Load and Score News

Place the FNSPID news file at `data/raw/fnspid_news.csv`. The expected columns are `headline`, `url`, `publisher`, `date`, and `stock`.

In [ ]:
NEWS_FILE = RAW / 'fnspid_news.csv'
TICKERS = ['AAPL', 'MSFT', 'GOOGL']

if not NEWS_FILE.exists():
    raise FileNotFoundError(
        f'Missing {NEWS_FILE}. Add the FNSPID news CSV before running this notebook.'
    )

news = load_news(NEWS_FILE)
news = news[news['stock'].isin(TICKERS)].copy()
news_sentiment = add_sentiment_scores(news)

news_sentiment[['headline', 'stock', 'date', 'sentiment_score', 'sentiment_label']].head()

In [ ]:
sentiment_summary = news_sentiment.groupby('sentiment_label').agg(
    articles=('headline', 'size'),
    avg_score=('sentiment_score', 'mean'),
).reindex(['negative', 'neutral', 'positive'])
sentiment_summary

## Load Historical Stock Prices

Local CSV files from `data/raw/prices/` are preferred for final reproducibility. If they are missing, the notebook downloads a reproducible date range from `yfinance`.

In [ ]:
price_frames = []
for ticker in TICKERS:
    price_file = RAW / 'prices' / f'{ticker}.csv'
    if price_file.exists():
        frame = load_price_data(price_file)
        source = 'local CSV'
    else:
        downloaded = yf.download(
            ticker,
            start='2020-01-01',
            end='2026-01-01',
            auto_adjust=False,
            progress=False,
        ).reset_index()
        frame = normalize_price_data(downloaded)
        source = 'yfinance'
    frame['stock'] = ticker
    frame['source'] = source
    price_frames.append(frame)

prices = pd.concat(price_frames, ignore_index=True)
prices[['stock', 'source', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']].head()

In [ ]:
price_quality = prices.groupby('stock').agg(
    rows=('Date', 'size'),
    start_date=('Date', 'min'),
    end_date=('Date', 'max'),
    missing_adj_close=('Adj Close', lambda s: s.isna().sum()),
)
price_quality

## Align News to Returns and Correlate

In [ ]:
analysis = sentiment_return_dataset(news_sentiment, prices)
analysis = analysis.sort_values(['stock', 'trading_day']).reset_index(drop=True)
analysis.head()

In [ ]:
corr = pearson_correlation(analysis)
print(interpret_correlation(corr))
print(f'Matched stock-day observations: {len(analysis):,}')
print(f'Article count in matched set: {analysis["article_count"].sum():,}')

## Visualize the Relationship

In [ ]:
fig, ax = plot_sentiment_return_scatter(analysis, corr)
save_current_figure(FIGURES / 'sentiment_return_scatter.png')
plt.show()

In [ ]:
category_returns = average_return_by_sentiment(analysis)
fig, ax = plt.subplots(figsize=(7, 4))
category_returns.plot(kind='bar', ax=ax, color=['tab:red', 'tab:gray', 'tab:green'])
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Average Daily Return by Sentiment Category')
ax.set_xlabel('Sentiment category')
ax.set_ylabel('Average daily return (%)')
save_current_figure(FIGURES / 'return_by_sentiment_category.png')
plt.show()
category_returns

In [ ]:
stock_level_corr = (
    analysis.groupby('stock')
    .apply(lambda df: pearson_correlation(df), include_groups=False)
    .rename('pearson_r')
    .reset_index()
)
stock_level_corr

## Interpretation

TextBlob was selected because it gives a reproducible polarity score for short headline text without requiring external model downloads. After scoring headlines, the analysis aligns weekend and holiday articles to the next available trading day so news and returns share the same stock-day unit of analysis.

Use the printed Pearson coefficient and the two plots above in the final report. A positive coefficient means more positive average news sentiment is associated with higher same-day returns; a negative coefficient means the opposite. If the value is near zero, headline polarity alone is not a strong same-day predictor in this sample.

Limitations: correlation is not causation, reactions may occur before publication or with a multi-day lag, market-wide shocks can dominate company-specific news, and repeated syndicated headlines can overweight one narrative. A stronger production strategy would test lagged returns, abnormal returns versus a benchmark, and out-of-sample performance.